# Module 9: Rare Events, Zero Inflation and When to Aggregate Up

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Orrindale Police Department has eight sworn officers and averages under one use
of force incident a month. Forty three percent of its months are zero.

The reflex when half a column is zero is to reach for a zero inflated model.
This module checks whether that reflex is right, finds that it is not, and then
takes up the problem that actually exists at an agency this size, which is not
a modelling problem at all.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]          # never fit on unfinished months


CALENDAR = pd.period_range("2019-01", "2026-04", freq="M").to_timestamp()


def counts(agency_id):
    """Monthly counts on a complete calendar, so a gap stays visible as missing."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series(d["n_uof"].values, dtype=float,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


print(f"{final['agency_id'].nunique()} agencies, {final['year_month'].nunique()} months")

In [ ]:
import statsmodels.api as sm
from scipy import stats


def panel(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month").reset_index(drop=True)
    d = d.assign(dt=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    return d.assign(mo=d["dt"].dt.month,
                    yr=((d["dt"].dt.year - 2019) * 12 + d["dt"].dt.month - 1) / 12.0)


o = panel("A006")                      # Orrindale, 8 sworn officers
y = o["n_uof"].values.astype(float)
print(f"  {len(y)} months, mean {y.mean():.3f}, max {int(y.max())}")
print(f"  months at zero: {int((y == 0).sum())} of {len(y)}, "
      f"which is {100 * (y == 0).mean():.1f} percent")

## 2. The question to ask before fitting anything

Zero inflation means **more zeros than the count distribution can produce**. It
does not mean many zeros. A Poisson with a small mean produces a great many
zeros all by itself, and the probability is one line of arithmetic.

In [ ]:
lam = y.mean()
print(f"  a Poisson with mean {lam:.3f} puts {100 * np.exp(-lam):.1f} percent "
      f"of its mass at zero")
print(f"  the data has                {100 * (y == 0).mean():.1f} percent\n")
print("  full comparison:")
for k in range(0, 6):
    print(f"    {k} incidents   observed {100 * (y == k).mean():5.1f}%   "
          f"Poisson {100 * stats.poisson.pmf(k, lam):5.1f}%")

**Forty three observed against forty five predicted.** There is no excess of
zeros to explain. The whole distribution matches a plain Poisson, which is what
the variance to mean ratio also says.

In [ ]:
print(f"  variance / mean = {y.var(ddof=1) / y.mean():.2f}   (1.00 is Poisson)")

## 3. Fitting the zero inflated model anyway

Worth doing once, so that the result is yours rather than an assertion.

In [ ]:
from statsmodels.discrete.count_model import ZeroInflatedPoisson

X = sm.add_constant(pd.DataFrame({
    "t": o["yr"].values,
    "sin1": np.sin(2 * np.pi * o["mo"].values / 12),
    "cos1": np.cos(2 * np.pi * o["mo"].values / 12),
}))
off = np.log(o["n_arrests"].values.astype(float))

po = sm.GLM(y, X, family=sm.families.Poisson(), offset=off).fit()
zi = ZeroInflatedPoisson(y, X, offset=off).fit(disp=False)

print(f"  Poisson          AIC {po.aic:.1f}")
print(f"  zero inflated    AIC {zi.aic:.1f}")
print(f"\n  the inflation coefficient: {zi.params[0]:.2f}  (p = {zi.pvalues[0]:.3f})")
print(f"  observed zeros {int((y == 0).sum())}, the plain Poisson model expects "
      f"{np.exp(-po.mu).sum():.1f}")

Identical AIC, and a ConvergenceWarning. The inflation coefficient has run off
to a large negative number with a p value of 0.99, which is the optimiser
saying **the inflation probability is zero and the parameter is not
identified.** The warning is not a nuisance to be silenced, it is the answer.
The extra machinery found nothing to do.

A zero inflated model is for a process where some periods cannot produce an
event at all: a facility closed, a programme not yet running, a category not
collected. **That is a statement about the world, and it needs a reason, not a
histogram.** Orrindale has no such reason. Its officers were on duty every one
of those 38 months and nobody used force.

## 4. The problem that is actually there

Orrindale's difficulty is not the shape of its distribution. It is that a
series averaging 0.8 a month carries almost no information about a trend.

In [ ]:
def detectable(agency_id):
    """The smallest annual trend this agency's series could distinguish from zero."""
    g = panel(agency_id)
    Xg = sm.add_constant(pd.DataFrame({"t": g["yr"].values}))
    m = sm.GLM(g["n_uof"].values.astype(float), Xg, family=sm.families.Poisson(),
               offset=np.log(g["n_arrests"].values.astype(float))).fit()
    return g["n_uof"].mean(), 100 * (np.exp(1.96 * m.bse["t"]) - 1)


rows = []
for aid, nm in [("A006", "Orrindale"), ("A011", "Dunmoor"), ("A005", "Kelsmoor"),
                ("A010", "Pinecrest"), ("A009", "Prairie County"), ("A004", "Millgate"),
                ("A003", "Havenbrook"), ("A001", "Stonewick"), ("A012", "Ashfell")]:
    mean, mde = detectable(aid)
    rows.append({"agency": nm, "incidents a month": round(mean, 2),
                 "smallest detectable annual trend": f"{mde:.1f}%"})
pd.DataFrame(rows).set_index("agency")

Every agency in the dataset is declining at about 4.9 percent a year. Stonewick
can see a trend of 1.3 percent, so it detects that decline comfortably.
**Orrindale cannot distinguish anything smaller than about 11.7 percent a
year**, so the real decline is invisible to it, and no choice of model changes
that. The information is not in the series.

This is the single most useful number to compute before promising an agency an
analysis. It takes four lines and it prevents a report that cannot be written.

## 5. Aggregating up

If monthly is too thin, the honest options are to widen the window or to widen
the group. Both trade resolution for signal, and both should be a stated
decision rather than a quiet one.

In [ ]:
s = o.set_index("dt")["n_uof"]
for rule, label in [("M", "monthly"), ("Q", "quarterly"), ("A", "annual")]:
    agg = s.resample(rule).sum()
    print(f"  {label:10s} {len(agg):3d} periods   mean {agg.mean():6.2f}   "
          f"zeros {100 * (agg == 0).mean():5.1f}%")

In [ ]:
small = final[final["agency_id"].isin(["A006", "A011"])]
pooled = small.groupby("year_month")[["n_uof", "n_arrests"]].sum().reset_index()
print(f"  the two smallest agencies pooled: mean {pooled['n_uof'].mean():.2f} a month, "
      f"{100 * (pooled['n_uof'] == 0).mean():.1f} percent zeros")

g = pooled.assign(dt=pd.PeriodIndex(pooled["year_month"], freq="M").to_timestamp())
g = g.assign(yr=((g["dt"].dt.year - 2019) * 12 + g["dt"].dt.month - 1) / 12.0)
mp = sm.GLM(g["n_uof"].values.astype(float), sm.add_constant(g[["yr"]].astype(float)),
            family=sm.families.Poisson(),
            offset=np.log(g["n_arrests"].values.astype(float))).fit()
print(f"  pooled, the smallest detectable annual trend is "
      f"{100 * (np.exp(1.96 * mp.bse['yr']) - 1):.1f} percent")

Pooling the two smallest agencies brings the threshold from 11.7 percent down
to 8.2, and the real decline of 4.9 percent is still below it. **Two small agencies together are still
small.** The genuine fix is the next module: put all twelve in one model and
let the common structure be estimated from all of them at once.

## 6. What to report for an agency this size

| Do | Do not |
|---|---|
| Report counts, and the period they cover | report a rate per 100 arrests for a single month |
| Give an interval, always | give a point estimate for a month with 1 incident |
| Say what the series cannot detect | say "no significant change" and stop |
| Aggregate to a year and say so | quietly switch the window when the answer is inconvenient |
| Compare against a pooled peer group | rank 300 agencies by a monthly rate |

The one line that matters most in a small agency report is the one stating what
could not have been seen. Without it, "no change was detected" reads as
evidence of no change.

## Exercise

Kelsmoor is the one agency in the dataset with more zeros than a Poisson
predicts. Run the section 2 check on it, and then decide what the excess
actually is.

In [ ]:
# Fill in the blank, then run.
AGENCY = None          # try "A005"

if AGENCY:
    g = panel(AGENCY)
    v = g["n_uof"].values.astype(float)
    print(f"  mean {v.mean():.3f}   variance / mean {v.var(ddof=1) / v.mean():.2f}")
    print(f"  zeros observed {100 * (v == 0).mean():.1f}%   "
          f"Poisson predicts {100 * np.exp(-v.mean()):.1f}%")
    print(f"  largest month: {int(v.max())} incidents\n")
    for k in range(0, 6):
        print(f"    {k}   observed {100 * (v == k).mean():5.1f}%   "
              f"Poisson {100 * stats.poisson.pmf(k, v.mean()):5.1f}%")
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A005"
```

Kelsmoor has 10.2 percent zeros against a predicted 6.2, which looks like
inflation. It is not. Look at the variance to mean ratio, which is 1.56, and at
the largest month, which is 10 against a mean of 2.8.

**The excess is at both ends.** Kelsmoor has more zeros than a Poisson allows
*and* more large months, which is what overdispersion looks like in a small
count series. A zero inflated model addresses one tail and leaves the other,
and it will report a spurious inflation probability while doing so.

The negative binomial handles both, and a negative binomial with an overall
mean of 2.8 puts a little over 10 percent of its mass at zero without any
inflation component at all.

**Excess zeros are the most frequently over diagnosed feature in count data.**
Check the upper tail before concluding anything about the lower one.

</details>

---

**Next:** [Module 10: Panel and Hierarchical Time Series](Module_10_Panel_And_Hierarchical.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*